# model-save-state-dict — faded example 2: Fill the atomic rename

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `model-save-state-dict`. Running the beacon reports progress on the `Distributed: model save state_dict rank-0` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: model save state_dict rank-0` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`model-save-state-dict`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "model-save-state-dict"
DD_SUBTOPIC = "Distributed: model save state_dict rank-0"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Atomic checkpointing writes to a tmp path then atomically renames it onto the final path with `os.replace(tmp, final)`, so readers never observe a partial file.

## Faded exercise 2

Complete `atomic_save(rank, dist_mod, model, path)`. The tmp write and barrier are given; fill in the atomic rename that promotes the tmp file to the final path.

**Fill in:** the os.replace call renaming tmp onto the final path

In [ ]:
import torch as t
import torch.nn as nn
import os

class MockDist:
    def __init__(self):
        self.barrier_calls = 0
    def barrier(self):
        self.barrier_calls += 1

def atomic_save(rank, dist_mod, model, path):
    if rank == 0:
        tmp = path + '.tmp'
        t.save(model.state_dict(), tmp)
        raise NotImplementedError()  # TODO: the os.replace call renaming tmp onto the final path
    dist_mod.barrier()
    return path


def _test():
    import tempfile, os
    t.manual_seed(42)
    model = nn.Linear(3, 3, bias=False)
    with t.no_grad():
        model.weight.fill_(2.0)
    tmp = tempfile.mkdtemp()
    path = os.path.join(tmp, 'm.pt')
    ret = atomic_save(0, MockDist(), model, path)
    assert ret == path
    assert os.path.exists(path)
    assert not os.path.exists(path + '.tmp'), 'tmp must be renamed away'
    assert t.load(path, weights_only=True)['weight'].sum().item() == 18.0


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn
import os

class MockDist:
    def __init__(self):
        self.barrier_calls = 0
    def barrier(self):
        self.barrier_calls += 1

def atomic_save(rank, dist_mod, model, path):
    if rank == 0:
        tmp = path + '.tmp'
        t.save(model.state_dict(), tmp)
        os.replace(tmp, path)
    dist_mod.barrier()
    return path
```
</details>